*© 2026 Paul Fergus. Free for student and research use — commercial use is strictly prohibited.*

# Lab 5 — Deep learning with custom image data

**Module:** Deep Learning Concepts and Techniques (Computer Vision)  
**Week:** 5  
**Estimated time:** 180 minutes

---

## Learning outcomes

By the end of this lab you should be able to:

1. Organise a folder of image files into the standard `train/val/test` × `class_name/` structure required by PyTorch's `ImageFolder` loader.
2. Build separate `transforms.Compose` pipelines for training and evaluation, and explain *why* augmentation only belongs on the training pipeline.
3. Use `ImageFolder` and `DataLoader` to feed batches of decoded JPEG/PNG images into a CNN.
4. Train a binary classifier on a real biomedical imaging task and evaluate it with metrics appropriate to a clinical screening problem.
5. Predict the class of an arbitrary image file given its path on disk — the canonical 'real world deployment' question.
6. Reason about which kinds of mistake matter most in a medical screening context.

## Prerequisites

- **Labs 1–4** completed.
- The textbook *Applied Deep Learning* (Fergus & Chalmers), Chapter 6 — applied computer vision case studies.
- Lecture 5: *From packaged datasets to deployment — loading and pre-processing real image data*.

## The thread from last week

In Labs 3 and 4 our datasets came packaged: one line of code (`datasets.MNIST(...)`, `datasets.CIFAR10(...)`) and `torchvision` handed us a tidy `(image, label)` pair. **Real projects don't start like that.** They start with someone handing you a folder of images organised into subfolders by class, or a CSV that maps filenames to labels, or worse, a flat directory and a separate spreadsheet. The first job is always the same: turn that mess into something a training loop can consume.

Today we learn the standard PyTorch pattern for the most common version of this: **one folder per class, one image file per sample**, using `torchvision.datasets.ImageFolder`.

## The task

We will train a CNN to classify microscope images of red blood cells as **parasitized** (infected with the *Plasmodium* malaria parasite) or **uninfected** (healthy). This is a binary classification problem with significant real-world precedent — malaria affects hundreds of millions of people annually and automated microscopy assistance is an active research area.

> **Note on the data.** This lab ships with a sample of 4,000 *synthetic* cell images that mimic the visual structure of Giemsa-stained blood smears — pink cells with optional dark parasite spots — but are not real biological samples. The pipeline, code, and evaluation are identical to what you'd use on the real **NIH Malaria Cell Images Dataset** (27,558 images), which is freely available from [the NIH cell-images page](https://ceb.nlm.nih.gov/repositories/malaria-datasets/) and [Kaggle](https://www.kaggle.com/datasets/iarunava/cell-images-for-detecting-malaria). To swap the real images in, follow `data/README.md`. **No code changes are required.**

## Useful references

- [`torchvision.datasets.ImageFolder` documentation](https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html)
- [Rajaraman, S. et al. (2018). *Pre-trained convolutional neural networks as feature extractors toward improved malaria parasite detection in thin blood smear images*](https://doi.org/10.7717/peerj.4568) — the paper that established the NIH dataset.

---

## 1. The dataset on disk

Before any code, let's look at how the files are laid out. The structure under `data/cell_images/` is the standard layout that PyTorch's `ImageFolder` understands without configuration:

```
cell_images/
├── train/
│   ├── parasitized/   ← class 0
│   │   ├── cell_00000.jpg
│   │   ├── cell_00001.jpg
│   │   └── ...
│   └── uninfected/    ← class 1
│       └── ...
├── val/
│   ├── parasitized/
│   └── uninfected/
└── test/
    ├── parasitized/
    └── uninfected/
```

**The rule is:** one folder per class, and a top-level `train/`, `val/`, or `test/` parent. `ImageFolder` reads the immediate subfolder name as the class label and walks the directory automatically. **Class labels are alphabetical**: `parasitized` becomes 0 and `uninfected` becomes 1. This will matter later when we interpret the confusion matrix.

Let's verify the folder counts match what we expect.

In [ ]:
import os
from pathlib import Path

DATA_DIR = Path("data/cell_images")

for split in ["train", "val", "test"]:
    for cls in ["parasitized", "uninfected"]:
        folder = DATA_DIR / split / cls
        n = len(list(folder.glob("*.jpg")))
        print(f"  {split:<5s} / {cls:<12s}: {n} images")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image

RNG_SEED = 7144
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {DEVICE}")

## 2. Looking at the images

Before we set up any pipeline, let's see what we're working with. Pulling a few images directly off disk with PIL — no PyTorch involved yet.

In [ ]:
para_files = sorted((DATA_DIR / "train" / "parasitized").glob("*.jpg"))
unin_files = sorted((DATA_DIR / "train" / "uninfected").glob("*.jpg"))

fig, axes = plt.subplots(2, 6, figsize=(13, 4.5))
for ax, fp in zip(axes[0], para_files[:6]):
    ax.imshow(Image.open(fp))
    ax.set_title("parasitized", fontsize=10)
    ax.axis("off")
for ax, fp in zip(axes[1], unin_files[:6]):
    ax.imshow(Image.open(fp))
    ax.set_title("uninfected", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

**What to observe.** The class signal is in the **dark spot(s)**: parasitized cells have one or more small dark purple/blue regions (mimicking *Plasmodium* parasites in a stained blood smear); uninfected cells are clean. Everything else — overall cell colour, position, brightness — varies within each class. The network's job is to learn 'spot detection' against irrelevant background variation.

(On the real NIH dataset, the parasites are more subtle and the background is messier. The model will work harder, but the pipeline is identical.)

### 2.1 Image dimensions — are they all the same size?

On packaged datasets like CIFAR-10, every image is 32×32. **On real datasets they almost never are.** Let's check our images.

In [ ]:
# Sample 200 training images and check their dimensions.
sample = []
for cls in ("parasitized", "uninfected"):
    files = list((DATA_DIR / "train" / cls).glob("*.jpg"))[:100]
    for fp in files:
        with Image.open(fp) as img:
            sample.append(img.size)   # (width, height)

widths, heights = zip(*sample)
print(f"Width:  min {min(widths)},  max {max(widths)},  mean {np.mean(widths):.1f}")
print(f"Height: min {min(heights)}, max {max(heights)}, mean {np.mean(heights):.1f}")

Our synthetic cells are all 130×130, but on the real NIH dataset you'd see considerable variation (from ~50×50 to ~300×300). **Either way the network needs a fixed input size**, so we'll resize every image to the same dimension as part of the transform pipeline. We'll use **130×130** to match the synthetic data; for the real dataset 130×130 is also a sensible average.

## 3. The transform pipelines

We will build **two** transform pipelines:

- **`train_transform`** — resize, augment, convert to tensor, normalise.
- **`eval_transform`** — resize, convert to tensor, normalise. *No* augmentation.

> ⚠️ **Important — a bug to learn from.** The original version of this lab applied the **same augmented `ImageDataGenerator`** to both the training and test data. That means every test image was randomly rotated, shifted, zoomed, and flipped before being shown to the model. This is a real bug with two consequences: (1) test performance becomes non-deterministic (different score every run), and (2) what you're measuring is no longer 'how well the model handles unseen images' — it's 'how well the model handles unseen *and randomly distorted* images', which is a different and easier-or-harder question depending on the distortion.
>
> **Augmentation goes on the training pipeline only.** Always.

Our augmentation here is mild: random horizontal flip and small rotation. We *do not* use vertical flip — for cell images that's defensible (cells have no canonical 'up'), but it's a thing to think about per dataset.

In [ ]:
IMAGE_SIZE = 130

# Modest mean and std — for the synthetic dataset, ImageNet-style stats also work fine.
# On the real NIH dataset you'd compute these from the training set.
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),                      # PIL -> tensor in [0, 1]
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

## 4. Loading the data with `ImageFolder`

Now the magic line — `ImageFolder` walks the directory and creates a `Dataset` for us:

In [ ]:
train_dataset = datasets.ImageFolder(root=DATA_DIR / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(root=DATA_DIR / "val", transform=eval_transform)
test_dataset = datasets.ImageFolder(root=DATA_DIR / "test", transform=eval_transform)

print(f"Train: {len(train_dataset)} samples")
print(f"Val:   {len(val_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")
print(f"Classes (in order): {train_dataset.classes}")
print(f"Class -> index map: {train_dataset.class_to_idx}")

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

**A small but important detail.** The class index mapping says `parasitized=0, uninfected=1`. That's alphabetical — `ImageFolder` doesn't know that 'parasitized' is the medically-positive case. **When we evaluate the model later, we have to remember which class is which.** A confusion matrix where row 0 is parasitized will look very different from one where row 0 is uninfected.

In [ ]:
# Show what augmentation does to one training image. Same image, different views.
def un_normalise(t: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
for ax in axes:
    img, lbl = train_dataset[0]    # different random transform every access
    ax.imshow(un_normalise(img))
    ax.set_title(train_dataset.classes[lbl], fontsize=10)
    ax.axis("off")
plt.suptitle("The same training image, 6 different augmentations", y=1.05)
plt.tight_layout()
plt.show()

## 5. The model

We reuse the **three-block CNN pattern** from Lab 4, adapted for 130×130×3 input instead of 32×32×3. The mechanics are identical; only the input size and the first dense layer's input dimensions change.

After three 2×2 pools, 130×130 → 65×65 → 32×32 → 16×16 (integer division). With 128 channels at the last conv layer, the flattened feature vector has size `128 × 16 × 16 = 32,768`.

In [ ]:
class CellCNN(nn.Module):
    """A three-block CNN for 130x130x3 image classification (binary)."""

    def __init__(self, n_classes: int = 2, dropout: float = 0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(2, 2)

        # 130 -> 65 -> 32 -> 16 after three 2x2 pools (integer division each time)
        self.flat_dim = 128 * 16 * 16
        self.fc1 = nn.Linear(self.flat_dim, 128)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))     # 130 -> 65
        x = self.pool(F.relu(self.bn2(self.conv2(x))))     # 65 -> 32
        x = self.pool(F.relu(self.bn3(self.conv3(x))))     # 32 -> 16
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


# Sanity-check shape flow before training.
_probe = CellCNN().to(DEVICE)
with torch.no_grad():
    xb, yb = next(iter(train_loader))
    out = _probe(xb.to(DEVICE))
print(f"Input batch shape: {xb.shape}")
print(f"Output shape:      {out.shape}  (expected (batch, 2))")
n_params = sum(p.numel() for p in _probe.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

**Note on architecture choice.** Two output units + `CrossEntropyLoss` is mathematically equivalent to one output unit + `BCEWithLogitsLoss` for binary classification. We use the two-output version here because it keeps the training-loop code identical to Labs 3 and 4. The choice doesn't change accuracy; it changes only how you index into the output to get a prediction.

## 6. Training

The same `EarlyStopping` and `run_epoch` helpers from Labs 2-4. By now this is muscle memory.

> ⏱ **Time.** On a typical laptop CPU, one epoch over 2,800 training images at 130×130 takes about 1 minute. Eight epochs ≈ 8 minutes. On GPU it's seconds.

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 3, min_delta: float = 0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.epochs_without_improvement = 0
        self.best_state = None
        self.should_stop = False

    def step(self, current_loss: float, model: nn.Module) -> None:
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.epochs_without_improvement = 0
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.epochs_without_improvement += 1
            if self.epochs_without_improvement >= self.patience:
                self.should_stop = True

    def restore_best(self, model: nn.Module) -> None:
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(mode=is_train)
    total_loss, total_correct, total_samples = 0.0, 0, 0
    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(1) == yb).sum().item()
            total_samples += xb.size(0)
    return total_loss / total_samples, total_correct / total_samples

In [ ]:
torch.manual_seed(RNG_SEED)
model = CellCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
early_stopping = EarlyStopping(patience=3)

MAX_EPOCHS = 8
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

t0 = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer)
    vl, va = run_epoch(model, val_loader, criterion, optimizer=None)
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl); history["val_acc"].append(va)
    print(f"epoch {epoch:>2d}  |  train loss {tl:.4f} acc {ta:.4f}  |  val loss {vl:.4f} acc {va:.4f}")
    early_stopping.step(vl, model)
    if early_stopping.should_stop:
        print(f"Early stopping at epoch {epoch}.")
        break

early_stopping.restore_best(model)
model.to(DEVICE)
print(f"\nDone in {(time.perf_counter() - t0)/60:.1f} min. Best val loss: {early_stopping.best_loss:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history["train_loss"], label="train", marker="o")
axes[0].plot(history["val_loss"], label="validation", marker="o")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history["train_acc"], label="train", marker="o")
axes[1].plot(history["val_acc"], label="validation", marker="o")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Test set evaluation

Now we touch the held-out test set, once. We report accuracy and a full classification report — and we **use the class names** so the confusion matrix doesn't show inscrutable integer labels.

In [ ]:
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(probs.argmax(axis=1))
        all_labels.append(yb.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)
y_prob = np.concatenate(all_probs)
class_names = train_dataset.classes      # ['parasitized', 'uninfected']

print(f"Test accuracy: {(y_pred == y_true).mean():.4f}\n")
print("Classification report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.show()

# Pull out the four cells using class_to_idx so we don't get confused.
para_idx = train_dataset.class_to_idx["parasitized"]
unin_idx = train_dataset.class_to_idx["uninfected"]

tp_para = cm[para_idx, para_idx]            # correctly called parasitized
fn_para = cm[para_idx, unin_idx]            # missed: had parasite, called clean
fp_para = cm[unin_idx, para_idx]            # false alarm: clean called parasitized
tn_para = cm[unin_idx, unin_idx]            # correctly called clean

print(f"\nCorrectly identified parasitized (TP):   {tp_para}")
print(f"Correctly identified uninfected (TN):    {tn_para}")
print(f"Healthy patient flagged as infected (FP): {fp_para}")
print(f"Infected patient missed (FN):             {fn_para}    ← the dangerous mistake")

## 8. Which error matters most?

Same kind of question we asked in Lab 2 with the breast cancer dataset, but the stakes here are also clinically severe. Two error types, two consequences:

- A **false positive** (healthy patient flagged as infected) sends them for confirmatory diagnosis — a finger prick, a microscope, perhaps unnecessary antimalarial medication. Stressful and costly but not life-threatening.
- A **false negative** (genuine infection missed) sends them home untreated. Untreated *P. falciparum* malaria can become fatal within 24 hours.

Just as in Lab 2, the errors are **asymmetric**. For a screening tool we strongly prefer high recall on the parasitized class, even at the cost of some extra false positives. Look at the per-class recall in your classification report — that's the number that matters most clinically.

> **A wider note on real medical AI.** A 99% test-accuracy model trained on cleanly-cropped, well-stained slides from one lab will *not* perform the same way on dirty smears from a field clinic in rural Uganda. The 'distribution shift' between training data and deployment data is one of the central unsolved problems in applied medical machine learning. We'll touch on this again in Lab 8.

## 9. Saving and loading the model

Same `state_dict` pattern as Lab 3. Save the numbers, keep the architecture in code.

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/cell_cnn.pt")
print("Saved to checkpoints/cell_cnn.pt")

# Reload into a fresh instance and verify it gives the same predictions.
fresh = CellCNN().to(DEVICE)
fresh.load_state_dict(torch.load("checkpoints/cell_cnn.pt", map_location=DEVICE, weights_only=True))
fresh.eval()
with torch.no_grad():
    sample_x = next(iter(test_loader))[0][:8].to(DEVICE)
    p1 = model(sample_x).argmax(1)
    p2 = fresh(sample_x).argmax(1)
print(f"Match: {torch.equal(p1, p2)}")

## 10. Predicting on a single image file

This is the 'deployed' use case: someone gives you a path to a cell image and asks for a prediction. You need to apply the **same preprocessing** the model saw during training (resize + normalise — but *not* augmentation, since we're at inference time).

In [ ]:
def predict_image(model, image_path: str | Path):
    """Predict the class of one image file. Returns (label, probability_distribution)."""
    img = Image.open(image_path).convert("RGB")
    x = eval_transform(img).unsqueeze(0).to(DEVICE)   # (1, 3, H, W)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    pred_idx = int(probs.argmax())
    return class_names[pred_idx], probs


# Pick one parasitized and one uninfected file from the test set.
para_test = next((DATA_DIR / "test" / "parasitized").glob("*.jpg"))
unin_test = next((DATA_DIR / "test" / "uninfected").glob("*.jpg"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, path, true_label in [(axes[0], para_test, "parasitized"), (axes[1], unin_test, "uninfected")]:
    pred, probs = predict_image(model, path)
    ax.imshow(Image.open(path))
    correct = (pred == true_label)
    title_colour = "#14532d" if correct else "#7a1f1c"
    ax.set_title(f"true: {true_label}\npredicted: {pred}  (p={probs.max():.3f})",
                 fontsize=11, color=title_colour)
    ax.axis("off")
plt.tight_layout()
plt.show()

---

## 11. Exercise 1 — image size and the speed/quality trade-off

Do **all three** parts.

**(a)** Re-train the model with `IMAGE_SIZE = 64` (smaller — much faster). You'll need to recompute the `flat_dim` in the model (after three 2×2 pools, 64 → 32 → 16 → 8, so it's `128 * 8 * 8 = 8192`). Record the wall-clock time per epoch and the final test accuracy.

**(b)** Re-train at `IMAGE_SIZE = 200` (larger — much slower). Recompute the flat dim again (200 → 100 → 50 → 25, so `128 * 25 * 25 = 80,000`). Note the parameter count of the dense classifier head ballooning. Wall-clock and accuracy?

**(c)** Plot the three accuracies against image size. Where does the curve flatten? Is bigger always better? Is there a 'sweet spot' for this task?

*Tip: write the model class with `flat_dim` computed from the input size, so you can swap sizes without manual recomputation. The pattern is:*

```python
spatial = IMAGE_SIZE // 2 // 2 // 2     # three pools
self.flat_dim = 128 * spatial * spatial
```

In [ ]:
# Your code for Exercise 1 (a), (b), (c) here.


*Your written observations for Exercise 1:*

(a) 

(b) 

(c) 

## 12. Exercise 2 — pick the right augmentations for *cells*

Augmentation appropriateness is dataset-specific. For each of the following transforms, decide whether it is **(✓) appropriate**, **(?) maybe**, or **(✗) inappropriate** for cell microscopy images, and **justify each in one sentence**. Then pick the three you most strongly endorsed, build a `train_transform_v2` that uses them all, retrain, and compare accuracy against the baseline.

1. `transforms.RandomHorizontalFlip(p=0.5)`
2. `transforms.RandomVerticalFlip(p=0.5)`
3. `transforms.RandomRotation(degrees=180)`
4. `transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05)`
5. `transforms.RandomGrayscale(p=0.2)`
6. `transforms.RandomErasing(p=0.3)`  *(applied after `ToTensor`)*
7. `transforms.GaussianBlur(kernel_size=3)`

**Hint.** Think about what *biological invariances* the model should learn. A cell rotated 180° is still a cell with a parasite. A cell with a `RandomErasing` patch over the parasite is *no longer a parasitized cell from the network's point of view*.

Write your appropriateness judgements *first*, then build the augmentation pipeline and run the experiment.

In [ ]:
# Your code for Exercise 2 here.


*Your appropriateness judgements (✓ / ? / ✗) with one-sentence justification each:*

1. RandomHorizontalFlip — 
2. RandomVerticalFlip — 
3. RandomRotation(180) — 
4. ColorJitter — 
5. RandomGrayscale — 
6. RandomErasing — 
7. GaussianBlur — 

*Your three chosen augmentations and the comparison result:*



## 13. Exercise 3 — pick one (open-ended)

**Pick *one* of the following options and complete it fully.**

### Option A — Threshold tuning for clinical safety

Since false negatives are the dangerous error, **shift the decision threshold** below 0.5 on the parasitized-class probability so the model flags more cases as parasitized. Specifically:

1. Compute predictions at thresholds 0.3, 0.4, 0.5, 0.6, 0.7 on the *probability of parasitized*.
2. For each, compute the recall on the parasitized class and the precision on the parasitized class.
3. Plot recall and precision against threshold.
4. State which threshold you would choose to deploy this model as a screening tool, and justify it.

### Option B — Class-imbalance handling

On the synthetic data the classes are balanced. The real NIH dataset is also fairly balanced, but many medical datasets are not. **Simulate class imbalance**: take only the first 200 parasitized training images (~14% of the original) but keep all uninfected images. Retrain the model. Does the model bias toward predicting the majority class? Now **fix it** in one of two ways: (i) by passing `class_weight` to `nn.CrossEntropyLoss(weight=...)`, or (ii) by using `WeightedRandomSampler` in your DataLoader. Compare. Which works better?

### Option C — Grad-CAM: what is the model looking at?

A medical AI that gets the right answer for the wrong reason is dangerous. **Grad-CAM** is a technique that highlights which regions of an image contributed most to a prediction. Implement Grad-CAM for the final conv layer (`conv3`) and visualise it on:

1. Two correctly-predicted parasitized images.
2. Two correctly-predicted uninfected images.
3. Two misclassified images.

Does the model attend to the parasites? Does it attend to *any* visually-meaningful region, or is it picking up spurious cues? Reflect in a paragraph.

*Grad-CAM in one paragraph: register a hook on `conv3` to capture its forward activations and its backward gradients during a single forward + backward pass. Multiply the activations by the channel-averaged gradients, ReLU the result, and resize to the input size for overlay. There are many tutorials online; a clean reference implementation is the [`pytorch-grad-cam`](https://github.com/jacobgil/pytorch-grad-cam) library if you'd rather use a library than write it from scratch.*

In [ ]:
# Your code for Exercise 3 (Option A, B, or C) here.


*Which option did you pick? Why?*

*Your written reflection (3–5 sentences):*



---

## 14. Reflection questions

Answer in the markdown cells below. Aim for 2–4 sentences per question.

**Q1.** Explain in plain English why we use **two** transform pipelines (a train one and an eval one), and what would go wrong if we used a single shared one.

**Q2.** `ImageFolder` assigns class indices alphabetically. Why does this matter, and what is a concrete bug that could happen if you forgot to check the class-to-index mapping?

**Q3.** The synthetic data in this lab is *much* cleaner than the real NIH dataset. Name **two** specific ways that the real-world distribution shift (between training data and deployment data) might make a model that scores 99% in the lab perform much worse in the field.

**Q4.** A colleague suggests using a much higher image resolution — say, 512×512 — to capture more detail in the parasite. Why might this actually *hurt* model performance for this task, given the size of the available training set? What is the underlying trade-off?

**Q5.** In the introduction we framed this as a 'screening tool' problem where high recall on the parasitized class matters most. Suggest **one** concrete way the model could be used *despite* not being perfect — i.e., describe a real deployment scenario in which a 95%-accurate model still adds clinical value.

*Your answers:*

**A1.** 

**A2.** 

**A3.** 

**A4.** 

**A5.** 

---

## What's next

In **Lab 6** we turn to a different kind of task: preparing data for something harder than classification. You'll annotate real images with **bounding boxes** for object detection, where the model has to say not just *what* is in the image but *where* it is, and *how many* — a fundamentally different problem from the single-label classifiers we've built in Labs 3-5. The annotated dataset you produce becomes the foundation for training an actual object detector in Lab 7.

Before leaving today, make sure:

- [ ] You have completed Exercises 1, 2, and 3
- [ ] You have answered the reflection questions
- [ ] Your notebook runs **top to bottom without errors** (*Kernel → Restart and Run All*)
- [ ] You have saved your work — the `labs/` folder is volume-mounted on your host